In [ ]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import sqlite3
from datetime import datetime

In [ ]:
db = sqlite3.connect("data/rased.db")
cursor = db.cursor()

In [ ]:
client = OpenAI(
    api_key="sk-GDsDnrnoU68uCBi2jhfC0A",
    base_url="https://elmodels.ngrok.app/v1"
)

In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-small")

index = faiss.read_index("data/faiss_index.bin")
with open("data/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

In [ ]:
def has_previous_records(cursor, patient_id):
    cursor.execute("SELECT COUNT(*) FROM visits WHERE patient_id = ?", (patient_id,))
    count = cursor.fetchone()[0]
    return count > 0

In [ ]:
def register_visit(cursor, db, patient_id, hospital_name, hospital_city):
    now = datetime.now()
    visit_date = now.strftime("%Y-%m-%d")
    visit_time = now.strftime("%H:%M:%S")

    cursor.execute("""
    INSERT INTO visits (patient_id, hospital_name, hospital_city, visit_date, visit_time)
    VALUES (?, ?, ?, ?, ?)
    """, (patient_id, hospital_name, hospital_city, visit_date, visit_time))

    db.commit()

    print("تم تسجيل حضور الموعد")

In [ ]:
# للم يعطي احتماليه انه الوقت غير منطقي
# rows[-1] == اخر موعد (الموعد الحالي)
# rows[:-1] == المواعيد السابقه

def analyze_visits(client, cursor, patient_id):
    cursor.execute("""
    SELECT * FROM visits
    WHERE patient_id = ?
    AND visit_date >= date('now', '-7 days')
    ORDER BY visit_date, visit_time
    """, (patient_id,))
    v_rows = cursor.fetchall()

    response = client.chat.completions.create(
        model="nuha-2.0",
        messages = [
            {
                "role": "user",
                "content": f"""
انت مكتشف احتيال استخدام هوية شخص اخر في المستشفى حيث يمكنك كشف هذا الاحتيال بناء على الزمن بين الموعد الحالي واي موعد سابق والاخذ بالحسبان المسافه بين المستشفيين

عليك اعطاء نسبه للموثوقية حيث ان 100 تعني موثوق تماماً و 0 تعني غير موثوق ابداً(منتحل)

الموعد الحالي: {v_rows[-1]}
الموعد السابقة: {v_rows[:-1]}

ملاحظات مهمة:
ارجع فقط نسبة الموثوقية لكل موعد سابق على حده مع ذكر سبب هذا التقييم
لا تفترض اخطاء في الادخال
لا ترجع اي كلام اضافي 
"""
            }
        ]
    )

    print(response.choices[0].message.content)

In [ ]:
def add_vitals(cursor, db, patient_id, visit_id, gender, height, weight):
    cursor.execute("""
    INSERT INTO vital_signs (patient_id, visit_id, entered_gender, height_cm, weight_kg, recorded_date)
    VALUES (?, ?, ?, ?, ?, DATE('now'))
    """, (patient_id, visit_id, gender, height, weight))

    db.commit()

In [ ]:
# اكتشاف الاحتيال من خلال الطول والوزن والجنس

def analyze_vitals(client, cursor, patient_id):
    cursor.execute("""
    SELECT 
        vs.entered_gender,
        vs.height_cm,
        vs.weight_kg,
        vs.recorded_date
    FROM vital_signs vs
    WHERE patient_id = ?
    ORDER BY recorded_date DESC
    LIMIT 2
    """, (patient_id,))
    vs_rows = cursor.fetchall()

    #if vs_rows[0][0] != vs_rows[1][0]:
    #    print("احتيال: بسبب اختلاف الجنس")

    response = client.chat.completions.create(
        model="nuha-2.0",
        messages = [
            {
                "role": "user",
                "content": f"""
انت مكتشف احتيال استخدام هوية شخص اخر في المستشفى حيث يمكنك كشف هذا الاحتيال بناء على تقييم الاختلاف بين كشفين للطول والوزن بناء على المدة بينمها 
عليك اعطاء نسبه للموثوقية حيث ان 100 تعني موثوق تماماً و 0 تعني غير موثوق ابداً(منتحل)

البيانات: {vs_rows}

ملاحظات مهمة:
ارجع فقط نسبة الموثوقية لكل من الطول والوزن على حده مع ذكر سبب هذا التقييم
لا تفترض اخطاء في الادخال او القياس
لا ترجع اي كلام اضافي 
"""
            }
        ]
    )

    print(response.choices[0].message.content)

In [ ]:
def add_diagnosis(cursor, db, patient_id, visit_id, diagnosis):
    cursor.execute("""
    INSERT INTO diagnoses (patient_id, visit_id, diagnosis)
    VALUES (?, ?, ?)
    """, (patient_id, visit_id, diagnosis))

    db.commit()

In [ ]:
def analyze_diagnosis(client, cursor, patient_id):
    cursor.execute("""
    SELECT
        diagnosis
    FROM diagnoses
    WHERE patient_id = ?
    """, (patient_id,))
    d_rows = cursor.fetchall()

    cursor.execute("""
    SELECT
        vs.entered_gender,
        vs.height_cm,
        vs.weight_kg,
        p.birth_year
    FROM visits v
    JOIN vital_signs vs ON v.visit_id = vs.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    WHERE v.patient_id = ?
    ORDER BY v.visit_date DESC, v.visit_time DESC
    LIMIT 1
    """, (patient_id,))

    all_rows = cursor.fetchall()

    from datetime import date

    def calculate_age(born):
        today = date.today()
        return today.year - born

    age = calculate_age(all_rows[0][3])

    def search(query, k=5):
        q = f"query: {query}"
        q_embedding = model.encode([q])

        D, I = index.search(np.array(q_embedding), k)

        results = [all_chunks[i] for i in I[0]]
        return results

    rag_results = search(d_rows)


# اكتشاف الاحتيال من خلال المرض
    response = client.chat.completions.create(
        model="nuha-2.0",
        messages = [
            {
                "role": "user",
                "content": f"""
 انت مكتشف احتيال استخدام هوية شخص اخر في المستشفى حيث يمكنك كشف هذا الاحتيال بناء على تقييم امكانية الاصابة بالمرض الحالي بناء على سجل الامراض السابقة وبيانات المريض واستنادا على المعلومات الطبية  
عليك اعطاء نسبه للموثوقية حيث ان 100 تعني موثوق تماماً و 0 تعني غير موثوق ابداً(منتحل)

البيانات: 
المرض الحالي: {d_rows[-1]}
الامراض السابقة: {d_rows[:-1]}
بيانات المريض: {all_rows[0][:3]}
عمر المريض: {age}
بيانات من قاعدة معلومات طبية: {rag_results}


ملاحظات مهمة:
ارجع فقط نسبة الموثوقية مع ذكر سبب هذا التقييم
لا تفترض اخطاء في الادخال او القياس
لا ترجع اي كلام اضافي 
"""
            }
        ]
    )

    print(response.choices[0].message.content)

In [ ]:
def main():
    import sqlite3

    db = sqlite3.connect("data/mirsad.db")
    cursor = db.cursor()

    print("راصد")

    patient_id = input("أدخل رقم المريض: ")

    hospital_name = input("أدخل اسم المستشفى: ")
    hospital_city = input("أدخل اسم المدينة: ")
    register_visit(cursor, db, patient_id, hospital_name, hospital_city)

    cursor.execute("""
    SELECT visit_id FROM visits
    WHERE patient_id = ?
    ORDER BY visit_id DESC
    LIMIT 1
    """, (patient_id,))
    
    visit_id = cursor.fetchone()[0]

    has_history = has_previous_records(cursor, patient_id)

    if not has_history:
        print(" لا يوجد سجل سابق → سيتم تخطي المقارنات")
    else:
        print("\n تحليل المواعيد:")
        result = analyze_visits(client, cursor, patient_id)
        print(result)

    print("\n إدخال العلامات الحيوية")
    gender = input("الجنس: ")
    height = float(input("الطول (سم): "))
    weight = float(input("الوزن (كجم): "))

    add_vitals(cursor, db, patient_id, visit_id, gender, height, weight)

    if has_history:
        print("\n تحليل العلامات الحيوية:")
        result = analyze_vitals(client, cursor, patient_id)
        print(result)

    print("\n إدخال التشخيص")
    diagnosis = input("أدخل التشخيص: ")
    add_diagnosis(cursor, db, patient_id, visit_id, diagnosis)

    if has_history:
        print("\n تحليل التشخيص:")
        result = analyze_diagnosis(client, cursor, patient_id)
        print(result)

    print("\n انتهت رحلة المريض بنجاح")


if __name__ == "__main__":
    main()